# 4. Controlling the search, and which classifier to use

The rules come from a genetic algorithm. This notebook shows the knobs
that matter (budget, early stopping, the objective, checkpoints, candidate
rules) and ends with a side-by-side comparison of the learners Ex-Fuzzy
offers, on the Wine data.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

from ex_fuzzy import BaseFuzzyRulesClassifier, FERL, classifiers, eval_rules, fuzzy_sets as fs, rule_mining, utils

wine = load_wine(as_frame=True)
X = wine.frame.drop(columns='target')
y = wine.target_names[wine.target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
partitions = utils.construct_partitions(X_train, fs.FUZZY_SETS.t1)
print(X.shape[1], 'features,', len(np.unique(y)), 'classes')


def report(name, model, seconds):
    return {'model': name, 'fit seconds': round(seconds, 1), 'test accuracy': round(model.score(X_test, y_test), 3)}

13 features, 3 classes


## Budget and early stopping

`n_gen` and `pop_size` set the budget. Early stopping ends the search when
the best fitness has not improved by `min_delta` for `patience`
generations; `patience=None` runs every generation.

In [2]:
rows = []
for patience in (None, 5):
    model = BaseFuzzyRulesClassifier(nRules=10, nAnts=3, linguistic_variables=partitions,
                                     n_gen=60, pop_size=30, patience=patience, random_state=0)
    start = time.perf_counter()
    model.fit(X_train, y_train)
    rows.append({'patience': 'None (run all)' if patience is None else patience, 'generations run': model.n_generations_run_,
                 **report('', model, time.perf_counter() - start)})
pd.DataFrame(rows).drop(columns='model')

,patience,generations run,fit seconds,test accuracy
0,None (run all),60,0.3,0.889
1,5,37,0.2,0.889


## The objective, and your own

The built-in objective maximises the Matthews correlation coefficient on
the training data, with optional penalties on the number of rules and
antecedents (`reparametrize_loss`). `customized_loss` replaces it with any
function of the rule base; it returns a value to maximise. Custom losses
run through the general evaluator, so they are slower than the built-in one.

In [3]:
def compact_accuracy(rule_base, X, y, tolerance, alpha, beta, precomputed_truth=None):
    """Accuracy minus a small price per rule."""
    evaluator = eval_rules.evalRuleBase(rule_base, X, y, precomputed_truth=precomputed_truth)
    evaluator.add_full_evaluation()
    return evaluator.acc - 0.01 * len(rule_base.get_rules())


custom = BaseFuzzyRulesClassifier(nRules=10, nAnts=3, linguistic_variables=partitions, n_gen=30, pop_size=30, random_state=0)
custom.customized_loss(compact_accuracy)
start = time.perf_counter()
custom.fit(X_train, y_train)
print(report('custom loss', custom, time.perf_counter() - start), '| rules:', len(custom.rule_base.get_rules()))

{'model': 'custom loss', 'fit seconds': 4.7, 'test accuracy': 0.852} | rules: 6


## Checkpoints

With `checkpoints=k` the PyMoo backend hands the best rule base every `k`
generations to a callback (or writes it to a file). Here the callback
records how the model grows during the search.

In [4]:
history = []
X_train_array = X_train.to_numpy()


def record(generation, rule_base):
    predicted = rule_base.winning_rule_predict(X_train_array, out_class_names=True)
    history.append({'generation': generation, 'rules': len(rule_base.get_rules()),
                    'train accuracy': round(np.mean(predicted == y_train), 3)})


tracked = BaseFuzzyRulesClassifier(nRules=10, nAnts=3, linguistic_variables=partitions, n_gen=30, pop_size=30, random_state=0, patience=None)
tracked.fit(X_train, y_train, checkpoints=5, checkpoint_callback=record)
pd.DataFrame(history)

,generation,rules,train accuracy
0,0,5,0.815
1,5,6,0.823
2,10,6,0.831
3,15,6,0.863
4,20,6,0.903
5,25,6,0.919


## Starting from mined candidate rules

`rule_mining` finds frequent, confident antecedent patterns per class.
Passing them as `candidate_rules` turns the search into a selection among
them, which is what `RuleMineClassifier` does in one call.

In [5]:
candidates = rule_mining.multiclass_mine_rulebase(X_train, y_train, partitions, support_threshold=0.05, max_depth=3)
print('candidate rules per class:', [len(base) for base in candidates])

selected = BaseFuzzyRulesClassifier(nRules=10, nAnts=3, linguistic_variables=partitions, n_gen=30, pop_size=30, random_state=0)
start = time.perf_counter()
selected.fit(X_train, y_train, candidate_rules=candidates)
print(report('selection among candidates', selected, time.perf_counter() - start), '| rules:', len(selected.rule_base.get_rules()))

candidate rules per class: [2360, 2822, 2166]


{'model': 'selection among candidates', 'fit seconds': 3.1, 'test accuracy': 0.963} | rules: 9


## Which classifier?

- `BaseFuzzyRulesClassifier`: the genetic search over rules, with fixed or
  optimised partitions. The reference method.
- `RuleMineClassifier`: mines candidates, then selects. Fast on wide data.
- `RuleFineTuneClassifier`: mines and selects, then a second search
  refines the selection.
- `FuzzyRulesClassifier`: a FARC-HD style association-rule classifier with
  feature capping and certainty-factor weights. Strong default accuracy.
- `FERL`: a greedy fuzzy rule tree with evidential outputs (notebook 6).

In [6]:
settings = dict(nRules=10, nAnts=3, n_gen=30, pop_size=30, random_state=0)
models = {
    'GA, fixed partitions': BaseFuzzyRulesClassifier(linguistic_variables=partitions, **settings),
    'GA, optimised partitions': BaseFuzzyRulesClassifier(**settings),
    'RuleMineClassifier': classifiers.RuleMineClassifier(linguistic_variables=partitions, **settings),
    'RuleFineTuneClassifier': classifiers.RuleFineTuneClassifier(linguistic_variables=partitions, **settings),
    'FuzzyRulesClassifier (FARC-HD)': classifiers.FuzzyRulesClassifier(nRules=10, random_state=0),
    'FERL': FERL(max_rules=10, random_state=0),
}


def rule_count(model):
    if isinstance(model, FERL):
        return model.n_rules()
    if isinstance(model, classifiers.FuzzyRulesClassifier):
        return model.n_rules_
    if hasattr(model, 'internal_classifier'):
        return len(model.internal_classifier().rule_base.get_rules())
    return len(model.rule_base.get_rules())


rows = []
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    rows.append({**report(name, model, time.perf_counter() - start), 'rules': rule_count(model)})
pd.DataFrame(rows)

,model,fit seconds,test accuracy,rules
0,"GA, fixed partitions",0.1,0.870,6
1,"GA, optimised partitions",0.8,0.852,4
2,RuleMineClassifier,4.3,0.907,10
3,RuleFineTuneClassifier,4.4,0.907,10
4,FuzzyRulesClassifier (FARC-HD),0.3,0.870,9
5,FERL,0.2,0.981,3


The numbers depend on the seed and the budget; the shape of the trade-off
does not. Mining-based learners are quick and compact, the genetic search
with optimised partitions is the most flexible, and FARC-HD is the safest
default when accuracy matters most.